# 12.3 怎麼看聲音隨時間變化？


一段聲音開始是低音，後來轉高音，只對整段統計會丟掉何時變化。我們可以像沿錄音移動一扇小窗，每次看一小段，叫時間框frame。相鄰框可以重疊，使變化不必等整框結束才被發現。本節先看哪些樣本被哪一框包含。

前置是[12.2樣本到時間的換算](https://birdhackor.github.io/tiny-perceptron-vlm/12.2.html)。框長400樣本，在16kHz下是0.025秒；移動步長hop為160樣本，即0.01秒。第一框取索引0到399，第二框160到559，所以兩框共用160到399這240個樣本。框長決定一回看多少資料，hop決定多久重新看一次。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
from tiny_perceptron.multimodal import tone

wave = tone()
frames = wave.unfold(0, 400, 160)
print("框形狀", tuple(frames.shape))
print("框秒數", 400 / 16000, "移動秒數", 160 / 16000)
print("重疊相同", torch.equal(frames[0, 160:], frames[1, :240]))

輸入是一段1600點錄音。`unfold(0,400,160)`沿時間軸0，每次取400點、向後160點；最後不足400的尾段不另成一框。輸出`(8,400)`，8框各400點；框0.025秒、移動0.01秒，最後True。比較式把第一框後240點與第二框前240點逐項核對，證明重疊不是複製了新聲音，而是重複觀察同一段樣本。

框數可手算為(1600−400)//160+1=8，最後一框從1120開始到1519。還剩80點沒有形成完整框，這個邊界選擇要記錄。後面的[STFT（short-time Fourier transform，短時傅立葉轉換）](https://birdhackor.github.io/tiny-perceptron-vlm/12.5.html)會把每個短框按頻率整理成強弱資訊，也就是逐框產生頻譜。這種工具預設可能在波形兩側補樣本，使第一個框以起點為中心，因此框數會不同；同一段聲音的兩種算法不能只比較形狀後就以為資料變長。

目前每框仍是400個原始振幅；後面會把每框轉成一組表示聲音的數字，例如各頻率的強弱，這組數字叫特徵。依照框的時間順序排列這些特徵，就是特徵序列；它的長度是框數8，400則是本例每框的原始樣本數。更多重疊會增加框數，也讓相鄰框共用較多內容。

訓練資料用來讓模型學習，測試資料用來檢查它能否處理沒學過的內容。不能把八框當成八段獨立錄音來拆到兩邊，否則幾乎相同的樣本會同時出現。應讓同一段錄音的重疊框一起放在同一邊，避免把接近看過的內容當新題。

可以把八框想成八張連續拍攝的近景照片，同一段物件會在相鄰照片裡重複出現。相鄰框共用內容，後續算出的特徵常會比較接近；這不保證每個特徵都平滑，也沒有讓原錄音的樣本總長度從1600變成3200。unfold產生的視圖可能共享原儲存，不應隨意對某框就地寫值，否則也可能影響重疊框；本例只讀它們來比較。若要保留尾端80點，有幾種選擇：補到完整框、接受較短末框或丟棄。不同選擇會改變後面特徵長度與邊界資訊，應在資料流程中寫明，而不是把不足一框的內容悄悄當成完整。

練習把`unfold(0, 400, 160)`的第三個參數改成400，也把印移動秒數的`160 / 16000`改成`400 / 16000`，讓輸出描述新的hop。將重疊比較那行換成`print("第一框末索引", 399, "第二框首索引", 400)`。先預測框形狀(4,400)、移動0.025秒，相鄰不再重疊，再執行核對。框長仍400，所以每一框分析範圍不變，改的是觀察頻率；紙上也可核對第一框0到399、第二框400到799，沒有共用樣本。
